<a href="https://colab.research.google.com/github/harshs-data/Pytorch/blob/main/Optimizing_the_Neural_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("zalando-research/fashionmnist")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'fashionmnist' dataset.
Path to dataset files: /kaggle/input/fashionmnist


In [3]:
import pandas as pd

# Assuming there is a file named 'fashion-mnist_train.csv' in that folder
# Adjust the filename based on what you saw in Step 2
df = pd.read_csv(f"{path}/fashion-mnist_train.csv")

# Look at the first 5 rows
df

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59995,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59996,1,0,0,0,0,0,0,0,0,0,...,73,0,0,0,0,0,0,0,0,0
59997,8,0,0,0,0,0,0,0,0,0,...,160,162,163,135,94,0,0,0,0,0
59998,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
# check for GPU

import torch

# This automatically picks GPU if available, otherwise stays on CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {device}")

Using device: cpu


In [5]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [6]:
# train test split
X_train, X_test, y_train, y_test =train_test_split(X,y, test_size=0.20, random_state=42)

In [7]:
X_train = X_train/255.0
X_test = X_test/255.0

In [24]:
class CustomDataset(Dataset):

  def __init__(self, features, labels):

    self.features = torch.tensor(features, dtype = torch.float32)
    self.labels = torch.tensor(labels, dtype=torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self, idx):
    return self.features[idx], self.labels[idx]



In [25]:
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)

In [26]:
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [27]:
len(train_dataloader) # no. of batches created

1500

In [28]:
class MyNN(nn.Module):

  def __init__(self, num_features):

    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(num_features, 128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(128,64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(64,10)
    )

  def forward(self, features):
    return self.model(features)


In [29]:
# imp. parameters
learning_rate=0.1
epochs=100

In [30]:
model = MyNN(X_train.shape[1])
model = model.to(device) # running model on GPU

In [31]:
# loss function
criterian = nn.CrossEntropyLoss()

In [32]:
# optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay=1e-4)


In [33]:
# training loop

for epoch in range(epochs):

  total_epoch_loss = 0

  for batch_features, batch_labels in train_dataloader:

    # Move data to GPU

    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)

    # forward pass
    outputs = model(batch_features)

    # loss
    loss = criterian(outputs, batch_labels)

    # back pass
    optimizer.zero_grad()
    loss.backward()

    # update weights
    optimizer.step()

    total_epoch_loss = total_epoch_loss + loss.item()

  avg_epoch_loss = total_epoch_loss/len(train_dataloader)

  print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_epoch_loss:.4f}")


Epoch 1/100, Loss: 0.6256
Epoch 2/100, Loss: 0.4965
Epoch 3/100, Loss: 0.4641
Epoch 4/100, Loss: 0.4350
Epoch 5/100, Loss: 0.4216
Epoch 6/100, Loss: 0.4052
Epoch 7/100, Loss: 0.3949
Epoch 8/100, Loss: 0.3823
Epoch 9/100, Loss: 0.3744
Epoch 10/100, Loss: 0.3696
Epoch 11/100, Loss: 0.3658
Epoch 12/100, Loss: 0.3586
Epoch 13/100, Loss: 0.3555
Epoch 14/100, Loss: 0.3512
Epoch 15/100, Loss: 0.3468
Epoch 16/100, Loss: 0.3466
Epoch 17/100, Loss: 0.3335
Epoch 18/100, Loss: 0.3362
Epoch 19/100, Loss: 0.3309
Epoch 20/100, Loss: 0.3286
Epoch 21/100, Loss: 0.3252
Epoch 22/100, Loss: 0.3212
Epoch 23/100, Loss: 0.3173
Epoch 24/100, Loss: 0.3222
Epoch 25/100, Loss: 0.3196
Epoch 26/100, Loss: 0.3169
Epoch 27/100, Loss: 0.3113
Epoch 28/100, Loss: 0.3144
Epoch 29/100, Loss: 0.3091
Epoch 30/100, Loss: 0.3015
Epoch 31/100, Loss: 0.3069
Epoch 32/100, Loss: 0.3033
Epoch 33/100, Loss: 0.3070
Epoch 34/100, Loss: 0.2974
Epoch 35/100, Loss: 0.2995
Epoch 36/100, Loss: 0.2995
Epoch 37/100, Loss: 0.2953
Epoch 38/1

In [33]:
# set model to eval model
model.eval()

In [34]:
# evaluation code

total = 0
correct = 0

for batch_features, batch_labels in test_dataloader:

  # move to GPU
  batch_features = batch_features.to(device)
  batch_labels = batch_labels.to(device)

  outputs = model(batch_features)

  _,predicted = torch.max(outputs,1)

  total = total + batch_labels.size(0)

  correct = correct + (predicted == batch_labels).sum().item()

print(f'correct: {correct}, total: {total}')
print(correct/total)


correct: 10371, total: 12000
0.86425


In [36]:
# evaluation code

total = 0
correct = 0

for batch_features, batch_labels in train_dataloader:

  # move to GPU
  batch_features = batch_features.to(device)
  batch_labels = batch_labels.to(device)

  outputs = model(batch_features)

  _,predicted = torch.max(outputs,1)

  total = total + batch_labels.size(0)

  correct = correct + (predicted == batch_labels).sum().item()

print(f'correct: {correct}, total: {total}')
print(correct/total)


correct: 43496, total: 48000
0.9061666666666667


### Basically optimization done to reduce the gap of accuracy between training evaluation and testing evaluation. Before this optimization test acc = 88% and training acc = 98%, which clearly indicates to Overfiting. So to reduce it we did 3 optimization technique (batch_norm, dropouts, and L2 regularization(weight_decay=1e-4)) by this gap bw both acc reduced from 10% to 4% gap.
